# 阶段七：失败、重试和幂等

只重试可恢复的临时错误，并用幂等键避免副作用重复执行。


In [ ]:
import asyncio


class TransientError(Exception):
    """表示可以有限重试的临时错误。"""


class PermanentError(Exception):
    """表示不应自动重试的永久错误。"""


async def run_with_retry(operation, retries: int, base_delay: float = 0.01) -> str:
    """执行可重试操作。

    参数:
        operation: 不接收参数、返回字符串的异步操作。
        retries: 允许的最大重试次数，不包含第一次执行。
        base_delay: 第一次重试前的等待秒数。

    返回:
        操作返回的字符串。

    异常:
        PermanentError 直接抛出；临时错误耗尽次数后抛出。
    """
    for attempt in range(retries + 1):
        try:
            return await operation()
        except PermanentError:
            raise
        except TransientError:
            if attempt == retries:
                raise
            await asyncio.sleep(base_delay * 2**attempt)
    raise AssertionError("unreachable")


async def query_tool(name: str, attempts: list[int]) -> str:
    """模拟查询工具；前两次调用临时失败。

    参数:
        name: 查询目标名称。
        attempts: 用于观察调用次数的可变计数器。

    返回:
        查询结果。
    """
    attempts[0] += 1
    if attempts[0] < 3:
        raise TransientError("temporary network error")
    return f"{name}: result"


async def idempotent_write(key: str, store: dict[str, str], attempts: list[int]) -> str:
    """模拟带幂等键的副作用写操作。

    参数:
        key: 本次业务操作唯一的幂等键。
        store: 保存已完成操作的字典。
        attempts: 用于观察实际副作用次数的可变计数器。

    返回:
        首次执行或重复执行时的稳定结果。
    """
    if key in store:
        return store[key]
    attempts[0] += 1
    store[key] = f"write accepted: {key}"
    return store[key]


async def langchain_retry_example() -> None:
    """使用 LangChain Runnable 的 with_retry 重试临时错误。"""
    from langchain_core.runnables import RunnableLambda

    attempts = [0]

    async def run(_: str) -> str:
        return await query_tool("langchain", attempts)

    runnable = RunnableLambda(func=lambda _: None, afunc=run).with_retry(
        retry_if_exception_type=(TransientError,), stop_after_attempt=3
    )
    result = await runnable.ainvoke("query")
    assert result == "langchain: result" and attempts[0] == 3
    print("LangChain Async:", result)


async def main() -> None:
    """验证原生重试、永久失败和幂等写入。"""
    attempts = [0]
    result = await run_with_retry(lambda: query_tool("native", attempts), retries=2)
    assert result == "native: result" and attempts[0] == 3
    print("native retry:", result, "attempts=", attempts[0])

    permanent_attempts = [0]

    async def invalid() -> str:
        permanent_attempts[0] += 1
        raise PermanentError("permission denied")

    try:
        await run_with_retry(invalid, retries=2)
    except PermanentError:
        assert permanent_attempts[0] == 1
        print("permanent error: not retried")

    store, writes = {}, [0]
    first = await idempotent_write("payment-1", store, writes)
    second = await idempotent_write("payment-1", store, writes)
    assert first == second and writes[0] == 1
    print("idempotent write:", second, "side_effects=", writes[0])
    await langchain_retry_example()


await main()


In [ ]:
from fastapi import FastAPI

app = FastAPI()


@app.post("/phase7/tools")
async def run_phase7_tools() -> dict[str, str]:
    """提供 HTTP 入口，复用重试查询和幂等写入。

    返回:
        查询结果和写入结果。
    """
    attempts = [0]
    query = await run_with_retry(lambda: query_tool("fastapi", attempts), retries=2)
    store, writes = {}, [0]
    write = await idempotent_write("request-1", store, writes)
    return {"query": query, "write": write}


assert any(route.path == "/phase7/tools" for route in app.routes)
print("FastAPI route registered")
